In [ ]:
%pip install scipy statsmodels

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from scipy import stats
import statsmodels.stats.proportion as smp

In [ ]:
import pandas as pd
import sqlite3
con = sqlite3.connect("C:/Users/fathi/Downloads/ipl sem 3/IPL-Cricket-Analyticss/data/raw/ipl.db")
def q(sql):
    return pd.read_sql(sql,con)

In [ ]:
toss_test=q("""
SELECT
      COUNT(*) AS n,
      SUM(CASE
          WHEN toss_winner=match_winner THEN 1
          ELSE 0 END) AS k
FROM matches_clean
WHERE result='win';
""")
toss_test

,n,k
0,1187,613


In [ ]:
result=stats.binomtest(
    k=613,
    n=1187,
    p=0.5)
result

BinomTestResult(k=613, n=1187, alternative='two-sided', statistic=0.5164279696714406, pvalue=0.2700389712415702)

In [ ]:
ci_low,ci_high=smp.proportion_confint(
    count=613,
    nobs=1187,
    alpha=0.05,
)

ci_low,ci_high

(0.48799919141387527, 0.5448567479290058)

Confidence Interval

Gives us a range of believable values for the underlying value based on sample data.

In our IPL data, the toss-winner is 51.64% of decisive matches. There is our observed proportion (point estimate). Instead of depending only on the single value, we calculate a confidence interval to show uncertainty in our estimate.

T TEST

In [ ]:
toss_decision_table = q("""

SELECT

    CASE
        WHEN toss_decision = 'bat' THEN 'bat'
        WHEN toss_decision = 'field' THEN 'field'
        ELSE 'unknown'
    END AS toss_decision,

    SUM(CASE
        WHEN toss_winner = match_winner THEN 1
        ELSE 0
    END) AS toss_winner_won,

    SUM(CASE
        WHEN toss_winner != match_winner THEN 1
        ELSE 0
    END) AS toss_winner_didnt_win,

    COUNT(*) AS total_matches

FROM matches_clean

WHERE result = 'win'

GROUP BY toss_decision

ORDER BY toss_decision;

""")

In [ ]:
toss_decision_table

,toss_decision,toss_winner_won,toss_winner_didnt_win,total_matches
0,bat,184,218,402
1,field,429,356,785


In [ ]:
table=[[184,218],
      [429,356]
]

In [ ]:
chi2,p,dof,expected=stats.chi2_contingency(table)
chi2,p,dof

(np.float64(8.04007460861096), np.float64(0.004575365240151297), 1)

In [ ]:
q("""
SELECT 
     COUNT(CASE WHEN win_by_runs IS NOT NULL THEN 1 END) AS run_wins,
     COUNT(CASE WHEN win_by_wickets IS NOT NULL THEN 1 END) AS wicket_wins
FROM matches_clean
WHERE result='win';
""")


,run_wins,wicket_wins
0,546,641


In [ ]:
q("""
SELECT
      CASE
      WHEN match_winner=innings1_batting_team
         THEN 'Defended'
      ELSE 'Chased Down'
      END AS outcome,
      COUNT(*) AS n,
      ROUND(AVG(first_innings_total),1) AS avg_first_innings_total
FROM v_match_totals
WHERE result='win'
AND first_innings_total IS NOT NULL
GROUP BY outcome;
""")

,outcome,n,avg_first_innings_total
0,Chased Down,647,155.5
1,Defended,540,183.3


In [ ]:
defended=q("""
SELECT first_innings_total
FROM v_match_totals
WHERE result='win'
AND first_innings_total IS NOT NULL
AND match_winner=innings1_batting_team;
""")['first_innings_total']

chased=q("""
SELECT first_innings_total
FROM v_match_totals
WHERE result='win'
AND first_innings_total IS NOT NULL
AND match_winner!=innings1_batting_team;
""")['first_innings_total']
len(defended),len(chased),defended.mean(),chased.mean()

(540, 647, np.float64(183.34074074074073), np.float64(155.53786707882534))

In [ ]:
t_stat,p_value=stats.ttest_ind(
    defended,
    chased,
    equal_var=False)
t_stat,p_value


(np.float64(15.730238965353694), np.float64(1.17272268263676e-50))

REJECT H0(NULL HYPOTHEIS)-There is statiscally significant evidence that the average first innings total differs between defended and chased down matches 

In [ ]:
with open ("../sql/02_category_cleaning.sql","r") as file:
    sql_script=file.read()
con.executescript(sql_script)
con.commit()


In [ ]:
q("""
SELECT
      bowler_type,
      COUNT(*) AS n
FROM v_deliveries_category_clean
GROUP BY bowler_type_clean
ORDER BY bowler_type_clean;""")

,bowler_type,n
0,,347
1,Left arm Fast,3727
2,Left arm Fast medium,15376
3,Left arm Medium,8778
4,Left arm Medium fast,13495
5,Left arm Wrist spin,4279
6,Legbreak,13893
7,Legbreak Googly,22670
8,Right arm Fast,42126
9,Right arm Fast medium,31227


In [ ]:
bowler_economy = q("""
SELECT bowler, bowler_type_clean,
    SUM(
        total_runs
        - COALESCE(bye_runs, 0)
        - COALESCE(leg_bye_runs, 0)
        - COALESCE(penalty_runs, 0)
    ) AS runs_conceded,
    
    SUM(
        CASE
            WHEN is_wide_ball = 0
            AND is_no_ball = 0
            THEN 1 ELSE 0
        END
    ) AS legal_balls

FROM v_deliveries_category_clean

WHERE bowler_type_clean IS NOT NULL
    AND is_super_over = 0

GROUP BY bowler, bowler_type_clean;
""")

bowler_economy.head()

,bowler,bowler_type_clean,runs_conceded,legal_balls
0,A Ashish Reddy,Right arm Medium,396,262
1,A Badoni,Right arm Offbreak,63,41
2,A Chandila,Right arm Offbreak,242,234
3,A Choudhary,Left arm Medium,144,101
4,A Dananjaya,"Right arm Offbreak, Legbreak",47,24


In [ ]:
qualified_bowlers=bowler_economy[
    bowler_economy['legal_balls']>=200].copy()
len(qualified_bowlers)

277

In [ ]:
def classify_bowler_type(x):
    x = str(x)

    # Spin style
    if ('spin' in x.lower()
        or 'legbreak' in x.lower()
        or 'offbreak' in x.lower()
        or 'orthodox' in x.lower()):
        return "Spin"

    # Pace styles
    elif 'fast medium' in x.lower():
        return "Medium-fast"
    elif 'medium fast' in x.lower():
        return "Medium-fast"
    elif 'fast' in x.lower():
        return "Fast"
    elif 'medium' in x.lower():
        return "Medium"

    return None


qualified_bowlers['group'] = qualified_bowlers['bowler_type_clean'].apply(
    classify_bowler_type
)

qualified_bowlers.groupby('group').size()

group
Fast           36
Medium         52
Medium-fast    96
Spin           93
dtype: int64

In [ ]:
bowler_economy['economy_rate']=(
    bowler_economy['runs_conceded']
    /bowler_economy['legal_balls']
    *6)
bowler_economy.head()

,bowler,bowler_type_clean,runs_conceded,legal_balls,economy_rate
0,A Ashish Reddy,Right arm Medium,396,262,9.068702
1,A Badoni,Right arm Offbreak,63,41,9.219512
2,A Chandila,Right arm Offbreak,242,234,6.205128
3,A Choudhary,Left arm Medium,144,101,8.554455
4,A Dananjaya,"Right arm Offbreak, Legbreak",47,24,11.750000


In [ ]:
qualified_bowlers.groupby('group')['economy_rate'].mean().round(3)


KeyError: 'Column not found: economy_rate'

In [ ]:
fast=qualified_bowlers.loc[
    qualified_bowlers['group']=='Fast','economy_rate']

medium=qualified_bowlers.loc[
    qualified_bowlers['group']=='Medium','economy_rate']

medium_fast=qualified_bowlers.loc[
    qualified_bowlers['group']=='Medium-fast','economy_rate']

spin=qualified_bowlers.loc[
    qualified_bowlers['group']=='Spin','economy_rate']

anova_result=stats.f_oneway(
    fast,medium,medium_fast,spin)
anova_result




F_onewayResult(statistic=np.float64(13.444563171677368), pvalue=np.float64(3.293037349980493e-08))

the four group annova found a statistically significant difference in economy rate among bowling types.group means suggest that spin bowlers drive this diffrence ,with a lower mean economy rate than the 3 pace group.

T test

In [ ]:
qualified_bowlers['pace_spin']=qualified_bowlers['group'].apply(
    lambda x:'Spin' if x=='Spin' else 'Pace')
qualified_bowlers.groupby('pace_spin').size()

pace_spin
Pace    184
Spin     93
dtype: int64

H0-mean economy rate is the same for pace bowlers and spin bowlers   
H1-mean economy rate is the different for pace bowlers and spin bowlers

In [ ]:
qualified_bowlers.groupby('pace_spin')['economy_rate'].agg(
    ['count','mean']).round(3)


KeyError: 'pace_spin'

T test

In [ ]:
spin_group =qualified_bowlers.loc[
    qualified_bowlers['pace_spin'] == 'Spin',
    "economy_rate"
]

pace_group = qualified_bowlers.loc[
    qualified_bowlers['pace_spin'] == 'Pace',
    "economy_rate"
]

pace_spin_test =stats.ttest_ind(
    spin_group,
    pace_group,
    equal_var= False
)

pace_spin_test

KeyError: 'pace_spin'

In [ ]:
import numpy as np
n_spin=len(spin_group)
n_space=len(pace_group)
pooled_sd=np.sqrt(
      ((n_spin-1)* spin_group.var(ddof=1)
       +(n_space -1)*pace_group.var(ddof=1))/(n_spin +n_space -2))
cohens_d=(
    pace_group.mean()-spin_group())/pooled_sd
pooled_sd,cohens_d
